## Indlæsning af data 

In [195]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [196]:
odr = pd.read_csv("data/age-dependency-ratio-old.csv")
schooling = pd.read_csv("data/average-years-of-schooling-among-adults.csv")
rd = pd.read_csv("data/research-spending-gdp.csv")
internet = pd.read_csv("data/share-of-individuals-using-the-internet.csv")
gdp_pc_long = pd.read_csv("data/gdp-per-capita-worldbank.csv")

In [197]:
odr = odr.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "odr"
})

schooling = schooling.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Both genders": "schooling"
})

rd = rd.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Research and development expenditure (% of GDP)": "rd"
})

internet = internet.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Share of the population using the Internet": "internet_use"
})

gdp_pc_long = gdp_pc_long.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "GDP per capita": "gdppc",
})

In [198]:
start = 2001
end = 2020

lagged_odr = odr.copy()

for df in [odr, gdp_pc_long, internet, rd, schooling]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

odr = odr[(odr["year"] >= start) & (odr["year"] <= end)]
gdp_pc_long = gdp_pc_long[(gdp_pc_long["year"] >= start) & (gdp_pc_long["year"] <= end)]
internet = internet[(internet["year"] >= start) & (internet["year"] <= end)]
rd = rd[(rd["year"] >= start) & (rd["year"] <= end)]
schooling = schooling[(schooling["year"] >= start) & (schooling["year"] <= end)]

### merge data

In [199]:
panel = internet[["country", "code", "year", "internet_use"]].merge(
    odr[["code", "year", "odr"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    rd[["code", "year", "rd"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    schooling[["code", "year", "schooling"]],
    on=["code", "year"],
    how="inner"
)

panel = panel.merge(
    gdp_pc_long[["code", "year", "gdppc"]],
    on=["code", "year"],
    how="inner"
)

panel["log_gdppc"] = np.log(panel["gdppc"])
panel["log_internet_use"] = np.log(panel["internet_use"])

print("Number of unique countries:", panel["code"].nunique())
print("Number of unique years:", panel["year"].nunique())

Number of unique countries: 140
Number of unique years: 20


### Balancer data

In [200]:
# Remove to countries that do not have data for all years
years_per_country = panel.groupby(["country", "code"])["year"].nunique().reset_index()
years_per_country = years_per_country.rename(columns={"year": "n_years"})
eligible_codes = years_per_country[years_per_country["n_years"] == 20]["code"]
panel_balanced = panel[panel["code"].isin(eligible_codes)].copy()


In [201]:
print("Number of unique countries:", panel_balanced["code"].nunique())
print("Number of unique years:", panel_balanced["year"].nunique())

print("\nCheck for NaN values:")
print(panel_balanced[["internet_use", "odr", "rd", "schooling", "gdppc"]].isna().sum())

Number of unique countries: 49
Number of unique years: 20

Check for NaN values:
internet_use    0
odr             0
rd              0
schooling       0
gdppc           0
dtype: int64


### Estimering af model

In [202]:
# kopi af det færdige balanced panel
df_est = panel_balanced.copy()

# estimation
model1= smf.ols(
    "log_internet_use ~ odr + C(code) + C(year)",
    data=df_est
)

results1 = model1.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results1 = pd.DataFrame({
    "coef": results1.params[["odr"]],
    "std_err": results1.bse[["odr"]],
    "p_value": results1.pvalues[["odr"]],
})

print(main_results1.round(4))

       coef  std_err  p_value
odr -0.1513   0.0398   0.0001


In [203]:
# kopi af det færdige balanced panel
df_est = panel_balanced.copy()

# estimation
model2 = smf.ols(
    "log_internet_use ~ odr + rd + C(code) + C(year)",
    data=df_est
)

results2 = model2.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results2 = pd.DataFrame({
    "coef": results2.params[["odr", "rd"]],
    "std_err": results2.bse[["odr", "rd"]],
    "p_value": results2.pvalues[["odr", "rd"]],
})

print(main_results2.round(4))

       coef  std_err  p_value
odr -0.1394   0.0349   0.0001
rd  -0.4234   0.1434   0.0032


In [204]:
# kopi af det færdige balanced panel
df_est = panel_balanced.copy()

# estimation
model3 = smf.ols(
    "log_internet_use ~ odr + rd  + log_gdppc + C(code) + C(year)",
    data=df_est
)

results3 = model3.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_est["code"]}
)

main_results3 = pd.DataFrame({
    "coef": results3.params[["odr", "rd", "log_gdppc"  ]],
    "std_err": results3.bse[["odr", "rd", "log_gdppc"]],
    "p_value": results3.pvalues[["odr", "rd", "log_gdppc"]],
})

print(main_results3.round(4))

             coef  std_err  p_value
odr       -0.1201   0.0319   0.0002
rd        -0.4322   0.1180   0.0002
log_gdppc  1.3079   0.2389   0.0000


In [207]:
results = summary_col(
    [results1, results2, results3],
    stars=True,
    model_names=['ODR', '+ RD', '+ log(GDPpc)'],
    regressor_order=['odr', 'rd', 'log_gdppc'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

#print the model
print("log(tfp) ~ odr + rd + log_gdppc + FE(code) + FE(year)")
print(results)

log(tfp) ~ odr + rd + log_gdppc + FE(code) + FE(year)

                  ODR        + RD    + log(GDPpc)
-------------------------------------------------
odr            -0.1513*** -0.1394*** -0.1201***  
               (0.0398)   (0.0349)   (0.0319)    
rd                        -0.4234*** -0.4322***  
                          (0.1434)   (0.1180)    
log_gdppc                            1.3079***   
                                     (0.2389)    
R-squared      0.8683     0.8796     0.9084      
R-squared Adj. 0.8585     0.8704     0.9013      
N              980        980        980         
R2             0.868      0.880      0.908       
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01
